## Imports

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split, cross_val_predict, GridSearchCV, RandomizedSearchCV
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.svm import LinearSVC, SVC
from sklearn.linear_model import LogisticRegression
from collections import Counter
from wordcloud import WordCloud
from sklearn.pipeline import Pipeline

## Load dataset

In [25]:
train_data = pd.read_csv('train.txt', sep='\t', header=None, names=['title', 'from', 'genre', 'director', 'plot'])

test_data = pd.read_csv('test_no_labels.txt', sep='\t', header=None, names=['title', 'from', 'director', 'plot'])

genres = list(train_data['genre'].unique())

## Pre-processing

In [35]:
stop_words = set(stopwords.words('english')) 
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = re.sub(r'\(\b([A-Z][a-z]*\s?)+\)', '', text)  # Remove text in brackets that starts with a capital letter (actors names)
    text = text.lower() 
    text = re.sub(r'http\S+', '', text) 
    text = re.sub(r"[^a-zA-Z+']", ' ', text)  # Keep only characters
    text = "".join([i for i in text if i not in string.punctuation])
    words = word_tokenize(text)
    text = " ".join([lemmatizer.lemmatize(i) for i in words if i not in stop_words])
    text = re.sub("\s[\s]+", " ", text).strip()  # Remove repeated/leading/trailing spaces
    return text

train_data['Text_cleaning'] = train_data['plot'].apply(clean_text) 
test_data['Text_cleaning'] = test_data['plot'].apply(clean_text)

train_data['Text_cleaning'].head()

0    sekhar graduate iim work business manager firm...
1    kris fowles go springwood diner meet ex boyfri...
2    cynthia traumatized death baby leaving bathtub...
3    four friend gangu abdul nihal gary form word g...
4    crisis middle class family son fall love fathe...
Name: Text_cleaning, dtype: object

## Statistics

In [ ]:
plt.figure(figsize=(10,6))
sns.countplot(y='genre' , data=train_data , order=train_data['genre'].value_counts().index, palette='viridis')
plt.title('Genre Distribution',fontweight='bold')
plt.xlabel('Count', fontweight='bold')
plt.ylabel('Genre', fontweight='bold')
plt.xticks(rotation=90)
plt.show()

### Frequency By Genre 

In [ ]:
for genre in genres:
    genre_data = train_data[train_data['genre'] == genre].copy()
    genre_data['words'] = genre_data['Text_cleaning'].apply(lambda x: x.split())

    # Concatenate all tokens for the genre
    all_words = [word for tokens in genre_data['words'] for word in tokens]

    # Get the most common words
    word_counts = Counter(all_words)

    print("-----", genre, "-----")
    print(word_counts.most_common(10))  # You can keep this line if you still want to print the counts

    # Create a word cloud
    wordcloud = WordCloud(width=800, height=400, background_color='white', colormap='viridis').generate_from_frequencies(word_counts)

    # Plot the word cloud
    plt.figure(figsize=(8, 4))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')  # Turn off axis
    plt.title(f'Word Cloud for {genre}', fontweight='bold')
    #plt.show()  # Show the word cloud


In [ ]:
from collections import Counter
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Combine all words across all genres
test_data['words'] = test_data['Text_cleaning'].apply(lambda x: x.split())  # Assuming 'Text_cleaning' column has cleaned text data
all_words = [word for tokens in test_data['words'] for word in tokens]

# Count the frequencies of the words
word_counts = Counter(all_words)

# Print the top 10 most common words
print("----- Overall Most Common Words in Test Data -----")
print(word_counts.most_common(10))

# Create the word cloud
wordcloud = WordCloud(width=800, height=400, background_color='white', colormap='viridis').generate_from_frequencies(word_counts)

# Plot the word cloud
plt.figure(figsize=(8, 4))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')  # Turn off axis
plt.title('Overall Word Cloud for Test Data', fontweight='bold')
plt.show()  # Show the word cloud

### Models

#### 1. Vectors

In [54]:
# Initialize the vectorizer stuff (possibel to use max_features=num_features)
tfidf_vectorizer = TfidfVectorizer(smooth_idf=True, ngram_range=(1, 3), use_idf=True, sublinear_tf=True, max_features=20000)
#tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, max_features=20000)
count_vectorizer = CountVectorizer()

# Fit and transform the text data in two flavours
X_train = tfidf_vectorizer.fit_transform(train_data['Text_cleaning'] + train_data['title'])
X_test = tfidf_vectorizer.transform(test_data['Text_cleaning'])
Z_train = count_vectorizer.fit_transform(train_data['Text_cleaning'])

#### 2. Train/test split

In [56]:
X = X_train
Z = Z_train
y = train_data['genre']
train_indices, val_indices, y_train, y_val = train_test_split(range(len(train_data)), y, test_size=0.1, random_state=40)

X_train, X_val = X[train_indices], X[val_indices]
Z_train, Z_val = Z[train_indices], Z[val_indices]

#### 3. K-fold Cross Validation

In [ ]:
folds = 8
model = clf

# Cross validation
cv_predictions = cross_val_predict(model, X, train_data['genre'], cv=folds)

print("WITH CROSS VALIDATION, WITH:", folds, "folds")
print("Accuracy: ", accuracy_score(train_data['genre'], cv_predictions))

##### SVM

In [57]:
import numpy as np
print("==== SVM ====")
clf = SVC(kernel='linear') #, class_weight='balanced')

print("Resultados com tfidf")
clf.fit(X_train, y_train)
x_pred=clf.predict(X_val)
print(classification_report(y_pred=x_pred, y_true=y_val, zero_division=1.))

# Identify misclassified examples and print the wrong predictions vs correct genres and their plots
print("\n==== Misclassified Predictions ====")
misclassified_indices = [i for i, (true, pred) in enumerate(zip(y_val, x_pred)) if true != pred]

for idx in misclassified_indices:
    print(f"\nPlot: {train_data.iloc[idx]['Text_cleaning']}")
    print(f"Correct genre: {y_val.iloc[idx]}, Predicted genre: {x_pred[idx]}")

#print("Resultados com count")
#clf.fit(Z_train, y_train)
#z_pred=clf.predict(Z_val)
#print(classification_report(y_pred=z_pred, y_true=y_val, zero_division=1.))

==== SVM ====
Resultados com tfidf
              precision    recall  f1-score   support

      action       0.70      0.68      0.69       111
   animation       0.92      0.74      0.82        73
      comedy       0.61      0.59      0.60       139
       crime       0.66      0.45      0.54        42
       drama       0.57      0.72      0.64       162
      horror       0.86      0.90      0.88       107
     romance       0.57      0.51      0.54        65
      sci-fi       0.73      0.53      0.62        15
     western       0.96      0.93      0.94        91

    accuracy                           0.71       805
   macro avg       0.73      0.67      0.69       805
weighted avg       0.72      0.71      0.71       805


==== Misclassified Predictions ====

Plot: cynthia traumatized death baby leaving bathtub accidentally drowned five friend jeff rob lynn paul terri decide go vacation take plane somewhere plagued engine trouble forced land plane lonely deserted island six set

##### Tuning of Hyperparameters

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(train_data['Text_cleaning'], train_data['genre'], test_size=0.1, random_state=40)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),  # TfidfVectorizer step
    ('svm', SVC())                 # SVM classifier step
])

# Define the parameter grid for both TfidfVectorizer and SVM
param_dist = {
    'tfidf__ngram_range': [(1, 1), (1, 2), (1, 3)],  # Test unigrams, bigrams, and trigrams
    'tfidf__max_df': [0.7, 0.85, 1.0],              # Ignore overly common words
    'tfidf__min_df': [1, 2, 5],                     # Ignore very rare words
    'tfidf__max_features': [5000, 10000, 20000],     # Limit the number of features
    'svm__kernel': ['linear', 'rbf'],        # SVM kernel types to try
    'svm__C': np.logspace(-2, 2, 5),                    # SVM regularization parameter
    'svm__gamma': ['scale', 'auto']                 # Kernel coefficient for rbf and poly kernels
}

# Set up GridSearchCV with the SVC model
grid_search = RandomizedSearchCV(estimator=pipeline, param_distributions=param_dist, n_iter=30,
                           scoring='accuracy', cv=3, n_jobs=-1, verbose=2)

# Fit the grid search to the training data
grid_search.fit(X_train, y_train)

# Get the best parameters and model
print(f"Best parameters found: {grid_search.best_params_}")
best_svm = grid_search.best_estimator_

# Predict and evaluate on validation set
y_pred = best_svm.predict(X_val)
print(classification_report(y_val, y_pred))

##### LinearSVC

In [ ]:
print("==== LinearSVC ====")
clf = LinearSVC() 

print("Resultados com tfidf")
clf.fit(X_train, y_train)
x_pred=clf.predict(X_val)
print("Validation Accuracy:", accuracy_score(x_pred, y_val))
print(classification_report(y_pred=x_pred, y_true=y_val, zero_division=1.))

print("Resultados com count")
clf.fit(Z_train, y_train)
z_pred=clf.predict(Z_val)
print(classification_report(y_pred=z_pred, y_true=y_val, zero_division=1.))

##### MultinomialNB

In [ ]:
print("==== MultinomialNB ====")
clf = MultinomialNB()

print("Resultados com tfidf")
clf.fit(X_train, y_train)
x_pred=clf.predict(X_val)
print(classification_report(y_pred=x_pred, y_true=y_val, zero_division=1.))

print("Resultados com count")
clf.fit(Z_train, y_train)
z_pred=clf.predict(Z_val)
print(classification_report(y_pred=z_pred, y_true=y_val, zero_division=1.))

##### GaussianNB (Requires .toarray(), kernel dies)

In [ ]:
print("==== GaussianNB ====")
clf = GaussianNB()

print("Resultados com tfidf")
clf.fit(X_train, y_train)
x_pred=clf.predict(X_val)
print(classification_report(y_pred=x_pred, y_true=y_val, zero_division=1.))

print("Resultados com count")
clf.fit(Z_train, y_train)
z_pred=clf.predict(Z_val)
print(classification_report(y_pred=z_pred, y_true=y_val, zero_division=1.))

##### LogisticRegression

In [ ]:
print("==== LogisticRegression ====")
clf = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)

print("Resultados com tfidf")
clf.fit(X_train, y_train)
x_pred=clf.predict(X_val)
print(classification_report(y_pred=x_pred, y_true=y_val, zero_division=1.))

print("Resultados com count")
clf.fit(Z_train, y_train)
z_pred=clf.predict(Z_val)
print(classification_report(y_pred=z_pred, y_true=y_val, zero_division=1.))

#### 7. Save model predictions

In [ ]:
# Use the trained model to make predictions on the test data
X_test_predictions = classifier.predict(X_test)

# Save the predictions to 'results.txt' with each predicted genre in a new line
with open('results.txt', 'w') as file:
    for genre in X_test_predictions:
        file.write(genre + '\n')

print("Predictions saved to results.txt")

### Bert

In [ ]:
from transformers import AlbertForSequenceClassification, AutoTokenizer
from torch.utils.data import DataLoader
from sklearn.preprocessing import LabelEncoder
import torch
from tqdm import tqdm
import torch.nn.functional as F
from torch.optim import AdamW

# Load the tokenizer for ALBERT
tokenizer = AutoTokenizer.from_pretrained("albert-base-v2")

# Encoding labels (if not already done)
label_encoder = LabelEncoder()
genres_as_integers = label_encoder.fit_transform(train_data['genre'])

# Train-validation split
X_train, X_val, y_train, y_val = train_test_split(train_data['plot'], genres_as_integers, test_size=0.2)

# Tokenize the data
train_encodings = tokenizer(X_train.tolist(), truncation=True, padding=True)
val_encodings = tokenizer(X_val.tolist(), truncation=True, padding=True)

# Dataset class remains unchanged
class MovieGenreDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

# Create train and validation datasets
train_dataset = MovieGenreDataset(train_encodings, y_train)
val_dataset = MovieGenreDataset(val_encodings, y_val)

# Set the device
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# Load ALBERT model for sequence classification
model = AlbertForSequenceClassification.from_pretrained('albert-base-v2', num_labels=len(genres_as_integers))
model.to(device)
model.train()

# Use a smaller batch size if needed
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# Optimizer
optim = AdamW(model.parameters(), lr=5e-5)

# Validation DataLoader
val_loader = DataLoader(val_dataset, batch_size=16)

# Function to calculate accuracy
def accuracy(outputs, labels):
    _, preds = torch.max(outputs, dim=1)
    return torch.tensor(torch.sum(preds == labels).item() / len(preds))

# Validation function
def validation(validation_dataloader):
    with torch.no_grad():
        loss_val_list = []
        preds_list = []
        accuracy_list = []
        accuracy_sum = 0
        for batch in tqdm(validation_dataloader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs[0]
            logits = F.softmax(outputs[1], dim=1)   # Taking the softmax of output
            _, preds = torch.max(logits, dim=1)      # Taking the predictions of our batch
            acc = accuracy(logits, labels)           # Calculating the accuracy of current batch
            accuracy_sum += acc                     # Taking sum of accuracies

            loss_val_list.append(loss)
            preds_list.append(preds)
            accuracy_list.append(acc)

        mean_accuracy = accuracy_sum / len(validation_dataloader)
        return mean_accuracy

# Training loop for ALBERT
for epoch in range(3):
    for batch in tqdm(train_loader):
        optim.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs[0]
        loss.backward()
        optim.step()
        optim.zero_grad()

model.eval()

# Run validation
validation(val_loader)


In [ ]:
print(validation(val_loader))

### Confusion matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Compute the confusion matrix
cm = confusion_matrix(y_val, y_pred, labels=genres)

# Display the confusion matrix
print("Confusion Matrix:")
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot()

# Show the plot
plt.show()

### LSTM

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# Tokenizing the text data (movie plots)
max_words = 5000  # Maximum number of words to keep in the tokenizer
max_len = 250     # Maximum plot length (truncate/pad all plots to this length)

plots = np.array(train_data['plot'])
genres_pd = np.array(train_data['genre'])

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(plots)

# Convert the plots to sequences of integers
sequences = tokenizer.texts_to_sequences(plots)

# Pad sequences to ensure uniform input size
X = pad_sequences(sequences, maxlen=max_len)

# Step 2: Encode genres as integers
genres = train_data['genre'].unique()
genre_to_id = {genre: idx for idx, genre in enumerate(genres)}
y = np.array([genre_to_id[genre] for genre in genres_pd])

# Convert the labels to one-hot encoding for multi-class classification
y = to_categorical(y, num_classes=len(genres))

# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 3: Build the LSTM model
model = Sequential()

# Add Embedding layer
model.add(Embedding(input_dim=max_words, output_dim=128, input_length=max_len))

# Add LSTM layer
model.add(LSTM(128, return_sequences=False))  # return_sequences=False since we're only interested in the final output

# Add Dropout for regularization
model.add(Dropout(0.2))

# Add a Dense layer for classification
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.2))

# Output layer with softmax for multi-class classification
model.add(Dense(len(genres), activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Step 4: Train the model
batch_size = 64
epochs = 10

history = model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs, validation_data=(X_val, y_val))

# Step 5: Evaluate the model
loss, accuracy = model.evaluate(X_val, y_val)
print(f'Validation Accuracy: {accuracy}')

# Save the model and tokenizer for later use
model.save('lstm_movie_genre_classifier.h5')

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import classification_report
from gensim.models import Word2Vec
from nltk.sentiment import SentimentIntensityAnalyzer
from scipy.sparse import hstack
from scipy.sparse import csr_matrix

# Uncomment these lines to download necessary NLTK resources if you haven't done so.
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('vader_lexicon')

# Load training data
train_data = pd.read_csv('train.txt', sep='\t', header=None, names=['title', 'from', 'genre', 'director', 'plot'])

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Clean the text
def clean_text(text):
    text = re.sub(r'\(\b([A-Z][a-z]*\s?)+\)', '', text)  # Remove actor names
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = "".join([i for i in text if i not in string.punctuation])
    words = word_tokenize(text)
    text = " ".join([lemmatizer.lemmatize(i) for i in words if i not in stop_words])
    text = re.sub("\s[\s]+", " ", text).strip()  # Remove repeated/leading/trailing spaces
    return text

train_data['Text_cleaning'] = train_data['plot'].apply(clean_text)

# Train Word2Vec model on the tokenized movie plots
tokenized_plots = [word_tokenize(plot.lower()) for plot in train_data['Text_cleaning']]
word2vec_model = Word2Vec(sentences=tokenized_plots, vector_size=200, window=4, min_count=1, workers=4)

# Function to convert a plot into a Word2Vec embedding
def plot_to_embedding(plot, model):
    words = word_tokenize(plot.lower())
    embeddings = [model.wv[word] for word in words if word in model.wv]
    if embeddings:
        return np.mean(embeddings, axis=0)
    else:
        return np.zeros(model.vector_size)

# Apply Word2Vec transformation to the cleaned plots
X_train_w2v = np.array([plot_to_embedding(plot, word2vec_model) for plot in train_data['Text_cleaning']])

# Initialize the TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(smooth_idf=True, ngram_range=(1, 2), use_idf=True, sublinear_tf=True, max_features=20000)
X_train_tfidf = tfidf_vectorizer.fit_transform(train_data['Text_cleaning'])

# Genre-Specific Frequent Words
genre_words = {
    'romance': ['love', 'family', 'life', 'friend', 'meet'],
    'horror': ['kill', 'death', 'house', 'body', 'find'],
    'action': ['police', 'kill', 'escape', 'fight', 'weapon'],
    'comedy': ['funny', 'laugh', 'friend', 'life', 'family'],
    'crime': ['police', 'murder', 'gang', 'detective', 'kill'],
    'drama': ['life', 'family', 'father', 'mother', 'find'],
    'sci-fi': ['alien', 'space', 'future', 'planet', 'ship'],
    'western': ['town', 'ranch', 'cowboy', 'gun', 'horse'],
    'animation': ['animal', 'child', 'adventure', 'magic', 'cartoon']
}

# Create genre-word-based features
def create_genre_word_features(plot, genre_words):
    features = {}
    for genre, words in genre_words.items():
        for word in words:
            features[f'{genre}_word_{word}'] = 1 if word in plot.lower() else 0
    return features

# Sentiment analysis features
sia = SentimentIntensityAnalyzer()

def create_sentiment_features(plot):
    sentiment = sia.polarity_scores(plot)
    return {
        'sentiment_pos': sentiment['pos'],
        'sentiment_neg': sentiment['neg'],
        'sentiment_neu': sentiment['neu'],
        'sentiment_compound': sentiment['compound']
    }

# Combine all features
def combine_features(X, X_w2v, genre_words):
    extra_features = []
    for plot in X:
        plot_features = {}
        plot_features.update(create_genre_word_features(plot, genre_words))
        #plot_features.update(create_sentiment_features(plot))
        extra_features.append(plot_features)
    
    vec = DictVectorizer()
    extra_features_vec = vec.fit_transform(extra_features)
    
    # Combine Word2Vec embeddings with extra features
    return hstack([X_w2v, extra_features_vec])

# Splitting the data into training and validation sets
y = train_data['genre']
train_indices, val_indices, y_train, y_val = train_test_split(range(len(train_data)), y, test_size=0.1, random_state=40)

# Apply combination of features only to the training set
X_train_combined = combine_features(train_data['Text_cleaning'].iloc[train_indices], X_train_w2v[train_indices], genre_words)

# Apply combination of features to the validation set
X_val_combined = combine_features(train_data['Text_cleaning'].iloc[val_indices], X_train_w2v[val_indices], genre_words)

# Convert to csr_matrix to ensure it is subscriptable
X_train_combined = csr_matrix(X_train_combined)
X_val_combined = csr_matrix(X_val_combined)

# Train the SVM model
clf = SVC(kernel='linear')
clf.fit(X_train_combined, y_train)

# Validate on the validation set
y_pred_val = clf.predict(X_val_combined)
print("Resultados com Word2Vec + Additional Features")
print(classification_report(y_pred=y_pred_val, y_true=y_val, zero_division=1.))

# For TF-IDF + Word2Vec combination (optionally)
X_train_combined_tfidf = hstack([X_train_tfidf[train_indices], X_train_combined])
X_val_combined_tfidf = hstack([X_train_tfidf[val_indices], X_val_combined])

# Re-train with TF-IDF and additional features (Word2Vec)
clf.fit(X_train_combined_tfidf, y_train)
y_pred_tfidf = clf.predict(X_val_combined_tfidf)
print("Resultados com TF-IDF + Word2Vec + Additional Features")
print(classification_report(y_pred=y_pred_tfidf, y_true=y_val, zero_division=1.))

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction import DictVectorizer
from scipy.sparse import hstack

# Assuming you have 'train_data' loaded with the 'Text_cleaning' and 'genre' columns

# Split the data before applying TF-IDF and feature extraction
X_train, X_val, y_train, y_val = train_test_split(train_data['Text_cleaning'], train_data['genre'], test_size=0.1, random_state=40)

# Genre-Specific Frequent Words
genre_words = {
    'romance': ['love', 'father', 'family', 'life', 'friend'],
    'horror': ['kill', 'death', 'house', 'body', 'find'],
    'action': ['police', 'killed', 'escape', 'fight', 'weapon'],
    'comedy': ['funny', 'laugh', 'friend', 'life', 'family'],
    'crime': ['police', 'murder', 'gang', 'detective', 'money'],
    'drama': ['life', 'family', 'father', 'mother', 'find'],
    'sci-fi': ['alien', 'space', 'future', 'planet', 'ship'], # 'earth'],
    'western': ['town', 'ranch', 'cowboy', 'gun', 'horse'],#, 'men'],
    'animation': ['animal', 'child', 'adventure', 'magic', 'cartoon'] #'bug', 'tom', 'jerry']
}

# Create genre-word features for each plot
def create_genre_word_features(plot, genre_words):
    features = {}
    for genre, words in genre_words.items():
        for word in words:
            features[f'{genre}_word_{word}'] = 1 if word in plot.lower() else 0
    return features

# Function to extract genre-word features
def extract_features(X, genre_words):
    extra_features = []
    for plot in X:
        plot_features = create_genre_word_features(plot, genre_words)
        extra_features.append(plot_features)

    vec = DictVectorizer()
    extra_features_vec = vec.fit_transform(extra_features)
    return extra_features_vec

# Initialize the TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(smooth_idf=True, ngram_range=(1, 2), use_idf=True, sublinear_tf=True, max_features=20000)

# Apply the vectorizer to the training and validation sets separately
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf = tfidf_vectorizer.transform(X_val)

# Extract additional genre-word features
X_train_genre_features = extract_features(X_train, genre_words)
X_val_genre_features = extract_features(X_val, genre_words)

# Combine TF-IDF and genre-word features for both training and validation sets
X_train_combined = hstack([X_train_tfidf, X_train_genre_features])
X_val_combined = hstack([X_val_tfidf, X_val_genre_features])

# Train the SVM model
svm = SVC(kernel='linear')
svm.fit(X_train_combined, y_train)

# Predict on the validation set
y_pred = svm.predict(X_val_combined)

# Evaluate the model
print(classification_report(y_val, y_pred))

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction import DictVectorizer
from scipy.sparse import hstack
import nltk
import spacy

# Download the necessary NLTK resources
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')  # Correct resource
nltk.download('vader_lexicon')

# Initialize spaCy's English model
nlp = spacy.load('en_core_web_sm')

# Example data (replace with your actual dataset)
# Assuming `train_data` has columns ['Text_cleaning', 'genre']
X_train, X_val, y_train, y_val = train_test_split(train_data['Text_cleaning'], train_data['genre'], test_size=0.1, random_state=40)

# 2. TF-IDF with N-grams (up to trigrams)
tfidf_vectorizer = TfidfVectorizer(smooth_idf=True, ngram_range=(1, 3), use_idf=True, sublinear_tf=True, max_features=20000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf = tfidf_vectorizer.transform(X_val)

# 3. Named Entity Recognition (NER)
def extract_ner_features(plot):
    doc = nlp(plot)
    entities = {}
    for ent in doc.ents:
        entities[f'entity_{ent.label_}_{ent.text}'] = 1
    return entities

# 4. Part of Speech (POS) Tagging
def extract_pos_features(plot):
    tokens = nltk.word_tokenize(plot)
    pos_tags = nltk.pos_tag(tokens)
    pos_counts = {}
    for word, tag in pos_tags:
        if tag not in pos_counts:
            pos_counts[tag] = 0
        pos_counts[tag] += 1
    return pos_counts

# Combine all feature extraction into a single function
def extract_combined_features(X):
    extra_features = []
    for plot in X:
        plot_features = {}
        # Add NER features
        plot_features.update(extract_ner_features(plot))
        # Add POS features
        plot_features.update(extract_pos_features(plot))
        extra_features.append(plot_features)
    
    vec = DictVectorizer()
    return vec.fit_transform(extra_features)

# Extract features for training and validation sets
X_train_combined_features = extract_combined_features(X_train)
X_val_combined_features = extract_combined_features(X_val)

# Combine TF-IDF features with the additional features
X_train_combined = hstack([X_train_tfidf, X_train_combined_features])
X_val_combined = hstack([X_val_tfidf, X_val_combined_features])

# Train the SVM model
svm = SVC(kernel='linear')
svm.fit(X_train_combined, y_train)

# Predict on the validation set
y_pred = svm.predict(X_val_combined)

# Evaluate the model
print(classification_report(y_val, y_pred))